In [ ]:
dnf install -y targetcli

In [ ]:
systemctl enable --now target

In [ ]:
targetcli

In [ ]:
/backstores/block create xdr-disk /dev/sdb

In [ ]:
/iscsi create iqn.2026-04.local.lab:xdr-storage

In [ ]:
/iscsi/iqn.2026-04.local.lab:xdr-storage/tpg1/luns create /backstores/block/xdr-disk

In [ ]:
cd /iscsi/iqn.2026-04.local.lab:xdr-storage/tpg1

In [ ]:
set attribute authentication=0

set attribute demo_mode_write_protect=0
set attribute generate_node_acls=1

In [ ]:
# For Delete lun
# /iscsi/iqn.2026-04.local.lab:xdr-storage/tpg1/luns delete lun0
# /backstores/fileio delete xdr-disk

In [ ]:
/iscsi/iqn.2026-04.local.lab:xdr-storage/tpg1/acls create iqn.1994-05.com.redhat:worker-1
/iscsi/iqn.2026-04.local.lab:xdr-storage/tpg1/acls create iqn.1994-05.com.redhat:worker-2
/iscsi/iqn.2026-04.local.lab:xdr-storage/tpg1/acls create iqn.1994-05.com.redhat:worker-3

In [ ]:
saveconfig
exit

On all Worker 

In [ ]:
dnf install -y iscsi-initiator-utils device-mapper-multipath

In [ ]:
systemctl enable --now iscsid
systemctl enable --now multipathd

In [ ]:
iscsiadm -m discovery -t sendtargets -p 172.16.6.69

In [ ]:
iscsiadm -m node --login

iscsiadm -m node -T iqn.2026-04.local.lab:xdr-storage -p 172.16.6.69:3260 -l

#iscsiadm -m node --logoutall=all
#iscsiadm -m node -o delete
# iscsiadm -m node -T iqn.2026-04.local.lab:xdr-storage -p 172.16.6.69:3260 -u


In [ ]:
cat > /etc/multipath.conf <<EOF
defaults {
    user_friendly_names yes
    find_multipaths yes
}
EOF

In [ ]:
systemctl restart multipathd

In [ ]:
multipath -ll

In [ ]:
lsblk

Peacmaker

In [ ]:
# For centos 
# sudo dnf config-manager --set-enabled highavailability

# For Redhat
# sudo subscription-manager repos --enable=rhel-9-for-x86_64-highavailability-rpms
dnf install -y pcs pacemaker corosync fence-agents-all

In [ ]:
systemctl enable --now pcsd
systemctl disable --now pcsd

In [ ]:
passwd hacluster
# the exact same password on all three VMs

In [ ]:
echo "1" | passwd --stdin hacluster

In [ ]:
mkdir -p /mnt/shared_data

On worker 2 only

In [ ]:
mkfs.xfs /dev/sdb

In [ ]:
pcs host auth worker1 worker2 worker3 -u hacluster

Setup and name the cluster (let's call it storage-cluster):

In [ ]:
pcs cluster setup storage-cluster worker1 worker2 worker3 --force

Start the cluster services across all nodes:

In [ ]:
pcs cluster start --all

Enable the cluster services to start on boot:

In [ ]:
pcs cluster enable --all

Disable fencing for the lab simulation:

In [ ]:
pcs property set stonith-enabled=false

- Enforce the "No Auto-Failback" rule:

This tells Pacemaker that if a resource moves due to a crash, it should stay exactly where it landed even when the broken node recovers.

In [ ]:
pcs resource defaults update resource-stickiness=100

Tell Pacemaker to manage your XFS formatted iSCSI disk. It will decide which single node gets to mount it.

In [ ]:
pcs resource create shared_storage Filesystem \
  device="/dev/sdb" \
  directory="/mnt/shared_data" \
  fstype="xfs" \
  op monitor interval=20s